# Exp 9 - reviewed population analysis

Pool only saved, reviewed individual-fish results. Detector parameters remain specific to each fish; this notebook never recomputes significant traces or overwrites individual caches. Fish with missing or inconsistent results appear explicitly in the readiness table. With one ready fish, every pooled figure is a **single-fish preview**, not an across-fish result.

Use the stable Cell 01-10 labels below when discussing this notebook; they do not depend on execution counts such as In[57]. Review context and the cell map are in [README.md](README.md).

## Cell 01 - Imports and shared helpers


In [ ]:
# Cell 01 - Imports and shared helpers
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/data_loading.py').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import src.several_fish_loading as loading
import src.several_fish_diagnostics as diagnostics
import src.several_fish_figures as figures
import src.several_fish_reporting as reporting

# Refresh edited helpers when rerunning this cell in an existing kernel.
# Compile current source directly so cached modules/bytecode cannot hide updates.
for module in (diagnostics, figures):
    module_source = REPO_ROOT / 'src' / (module.__name__.rsplit('.', 1)[-1] + '.py')
    exec(compile(module_source.read_text(encoding='utf-8-sig'),
                 str(module_source), 'exec'), vars(module))


## Cell 02 - Cohort, phase windows and display settings

Keep the same reliable-neuron population in every condition. The settings below explicitly define the two bout windows and the existing activity rule; no implicit calcium-response tail is added. Adjust fish selection after saving each fish's individual results.


In [ ]:
# Cell 02 - Cohort, phase windows and display settings
EXPERIMENT_NAME = 'Exp_9_RF_memory_inhibition'
LAB_ROOT = Path('D:/Alejandro/Data/OneDrive - Universit\u00e9 de Lausanne/Lab')
FISH_IDS = [f'L978_f{i:02d}' for i in range(1, 9)]
STIMULUS_ALIASES = {
    'LeC1': 'LeC1_Bout', 'LeC2': 'LeC2_Far',
    'LeS1': 'LeS1_Overlap', 'LeS2': 'LeS2_Overlap', 'LeS3': 'LeS3_Overlap',
    'LeS4': 'LeS4_Overlap', 'LeS5': 'LeS5_Overlap',
    'RiC1': 'RiC1_Bout', 'RiC2': 'RiC2_Far',
    'RiS1': 'RiS1_Overlap', 'RiS2': 'RiS2_Overlap', 'RiS3': 'RiS3_Overlap',
    'RiS4': 'RiS4_Overlap', 'RiS5': 'RiS5_Overlap',
}

COHORT_SETTINGS = dict(
    experiment_name=EXPERIMENT_NAME, fish_ids=FISH_IDS,
    main_path=LAB_ROOT / 'Data/2p', stimuli_main_path=LAB_ROOT / 'Data/stimuli',
    phase_metadata_path=LAB_ROOT / 'Results/Paper/2p_derived' / EXPERIMENT_NAME / 'stimuli/parameters/stimulus_time_summary.csv',
    stimulus_aliases=STIMULUS_ALIASES, selected_blocks=['B1', 'B2', 'B3', 'B4'],
    display_pre_s=5.0, recovery_s=10.0,
)
MEASUREMENT_SETTINGS = dict(
    bout_windows={'bout': (0.0, 3.6), 'bout_recovery': (0.0, 8.6)},
    pre_bout_s=2.0, active_fraction_threshold=0.10, min_epoch_s=1.0,
    min_active_reps=2, expected_reps=4,
)
SIDE_CONDITIONS = {side: [prefix + x for x in ['C1', 'C2', 'S1', 'S2', 'S3', 'S4', 'S5']]
                   for side, prefix in [('left', 'Le'), ('right', 'Ri')]}
DELAY_CONDITIONS = {side: conditions[2:] for side, conditions in SIDE_CONDITIONS.items()}
COMPARISONS = {f'{prefix}{condition}-C1': (prefix + condition, prefix + 'C1')
               for prefix in ['Le', 'Ri'] for condition in ['C2', 'S1', 'S2', 'S3', 'S4', 'S5']}
COMPARISONS.update({prefix + 'C2-S1': (prefix + 'C2', prefix + 'S1') for prefix in ['Le', 'Ri']})
RECRUITMENT_CONDITIONS = {side: [names[0], *DELAY_CONDITIONS[side]]
                          for side, names in SIDE_CONDITIONS.items()}  # C1 + S1-S5, no C2
RECRUITMENT_SETTINGS = dict(repetitions=(2, 3, 4), min_active_reps=2, bout_window='bout',
                            pre_active_fraction=0.75)  # ceil(0.75 * 15) = 12 passing trials
SORT_MODE = 'corravg'  # Pearson correlation, average-linkage clustering
SORT_STIMULUS_KEYS = ['LeC1_Bout', 'RiC1_Bout']
SORT_ALIGNMENT = 'bout'
SORT_WINDOW_S = (0.0, 3.6)  # nominal display times, excluding recovery
DISPLAY_SORT_SCOPES = ['within_fish', 'pooled']
DISPLAY_REPETITIONS = (2, 3, 4)  # None: all trials; (2, 3, 4): omit first, Cell 04 only
Z_LIMITS = (-1.0, 4.0)
ANALYSIS_PATH = Path(os.environ['SOCIAL_FILTERS_POPULATION_REPORT_ROOT']) if os.environ.get('SOCIAL_FILTERS_POPULATION_REPORT_ROOT') else LAB_ROOT / 'Results/Paper/2p_derived'

## Cell 03 - Readiness and saved settings

Missing fish are not silently regenerated. Invalid results are reported with their reason. The loader matches the saved session, timestamps, detector settings and original retained-neuron IDs. Older phase caches can read detector settings from their saved configuration.


In [ ]:
# Cell 03 - Readiness and saved settings
cohort = loading.load_reviewed_phase_cohort(COHORT_SETTINGS)
display(cohort['readiness'])
if not cohort['fish_ids']:
    raise ValueError('No reviewed fish are ready. Review/save individual outputs listed above first.')
print(f"Included {len(cohort['fish_ids'])} fish: {', '.join(cohort['fish_ids'])}")
if len(cohort['fish_ids']) == 1:
    print('SINGLE-FISH PREVIEW: across-fish variation is not yet available.')
display({fid: fish['detector_settings'] for fid, fish in cohort['all_fish_data'].items()})
FIGURES = {}

## Cell 04 - Pooled rasters and equally weighted fish means

Display only saved reliable neurons. `DISPLAY_REPETITIONS` in Cell 02 selects
one-based trials for these plots: `(2, 3, 4)` omits the first; `None` uses all.
This affects both heatmaps and fish mean traces, not Cell 05-09 measurements.
The retained-neuron population and saved detector results remain fixed.

For each alignment, show both within-fish sorting (cyan fish boundaries) and
pooled correlation sorting across fish (no fish grouping). Both use the same
all-trial C1-only bout z-score reference (left and right C1, 0-3.6 s) to keep neuron order fixed across
conditions, alignments and repetition selections. Original fish/neuron identities
remain in each display's `displayed_row_metadata`.

Significant rasters use white for zero and black for one. Trial averages show
activation fractions across selected trials; concatenated plots retain binary
frames and their original repetition numbers. Z-score heatmaps also use grayscale, with larger z-scores darker and the same
fixed `Z_LIMITS` in every panel. Mean traces average neurons within fish first, then give
each fish equal weight; they are shown once per alignment, since sorting does
not affect means. Phase shading uses nominal trajectory offsets; measurements
retain exact event-relative sample times.

**How the rows are sorted:** for each neuron, average all saved repetitions of
left C1, take nominal bout samples in [0, 3.6) s, then append the corresponding
right C1 samples. Compute Pearson correlation between these neuron profiles.
Average-linkage clustering uses distance 1 - correlation, placing similar
profiles near each other. This sorts response shapes, not response magnitude;
it is not a correlation between stimulus conditions. The resulting order is
reused in every condition and both trace types. All retained neurons stay;
constant/nonfinite profiles cannot be correlation-clustered and remain unsorted.
Only row order and colors change; measurements and normalization do not.


In [ ]:
# Cell 04 - Pooled rasters and equally weighted fish means
ZSCORE_CMAP = 'Greys'  # larger z-score = darker; preserve Z_LIMITS
RASTER_CMAP = 'Greys'  # white = no significant activity; black = activity
RASTER_FIGSIZE = (20, 9)
MEAN_FIGSIZE = (20, 7)
displays = {}
for alignment in ['sequence', 'bout']:
    for sort_scope in DISPLAY_SORT_SCOPES:
        phase_display = diagnostics.build_phase_population_display(
            cohort, alignment, sort_mode=SORT_MODE, sort_scope=sort_scope,
            repetitions=DISPLAY_REPETITIONS, sort_stimulus_keys=SORT_STIMULUS_KEYS,
            sort_alignment=SORT_ALIGNMENT, sort_window_s=SORT_WINDOW_S)
        displays[alignment, sort_scope] = phase_display
        for trace_type in ['zscore', 'raster']:
            for combine_mode in ['mean', 'concat']:
                prefix = '' if sort_scope == 'within_fish' else 'pooled_'
                name = f'{prefix}{alignment}_{trace_type}_{combine_mode}'
                FIGURES[name], _ = figures.plot_phase_cohort_rasters(
                    phase_display, trace_type, combine_mode, figsize=RASTER_FIGSIZE,
                    z_limits=Z_LIMITS, raster_cmap=RASTER_CMAP, zscore_cmap=ZSCORE_CMAP)
                plt.show()
    displays[alignment] = displays.get((alignment, 'within_fish'), phase_display)
    FIGURES[f'{alignment}_fish_means'], _ = figures.plot_phase_cohort_means(
        phase_display, figsize=MEAN_FIGSIZE)
    plt.show()


## Cell 05 - Native-sample phase measurements

Windows use each repetition's actual sample times relative to its logged event, without interpolation. AUC is signed z-score seconds. Peak and AUC are computed per neuron/repetition, then averaged within neurons and fish. Coverage tables record requested bounds, sample counts and first/last samples. Missing measurements are not zero.

The activity rule requires the configured significant-frame fraction AND consecutive epoch within a trial, then at least 2/4 passing trials. This describes activity **during** a window, including possible preconditioning carryover; it does not identify a causal bout response. C1's absent preconditioning is not applicable.


In [ ]:
# Cell 05 - Native-sample phase measurements
analysis = diagnostics.build_phase_population_analysis(cohort, MEASUREMENT_SETTINGS, COMPARISONS)
display(analysis['window_coverage'].groupby(['window', 'applicable'], dropna=False)['n_samples'].agg(['min', 'max']))
display(analysis['fish_summary'][['fish_id', 'condition', 'window', 'n_neurons', 'n_valid_active', 'active_fraction']].head(12))

## Cell 06 - Bout AUC, peak and recruitment

Each colored point/line is one fish; the black line weights fish equally. The denominator is the same saved reliable-neuron population for all conditions. Fractions with incomplete classifications remain missing and carry explicit valid counts.


In [ ]:
# Cell 06 - Bout AUC, peak and recruitment
for window in MEASUREMENT_SETTINGS['bout_windows']:
    for metric, units in [('auc_z_s', 'AUC (z-score s)'), ('peak_z', 'Peak z-score'),
                          ('active_fraction', 'Active-neuron fraction')]:
        name = f'{window}_{metric}'
        FIGURES[name], _ = figures.plot_phase_fish_summary(
            analysis['fish_summary'], metric, SIDE_CONDITIONS, cohort['fish_ids'],
            window, units, figsize=(14, 4.5))
        plt.show()

## Cell 06b - Shared and newly recruited neurons, repetitions 2-4

Significant traces only; C2 is excluded. Classify each phase using the Cell 05
native-window decisions (at least 10% significant frames AND a 1-second epoch),
requiring at least 2 of repetitions 2, 3 and 4. Bout measurements use [0, 3.6) s.
Shared means active in C1 and the condition's bout. New after preconditioning
means bout-active but inactive during that condition's own preconditioning.

Each side has one preconditioning panel (equal mean across S1-S5 and the three
selected repetitions), followed by C1 and S1-S5 bout panels. The same neuron
stays on the same row. Rows group the stable preconditioning pool first, then C1-active neurons,
additional bout-active neurons by first S condition, and the remainder.
The preconditioning pool requires activity in at least 75% of the 15 selected
presentations: **12/15** passing trials. The green strip marks this stable pool;
bout strips mark C1 overlap, new-after-preconditioning, or both. Classification
uses each individual condition, not the averaged preconditioning display.
Missing classifications remain missing; C1 has no preconditioning.

Count bars give equal-fish means; connected points show each fish. Pooled-row
identities, per-fish fractions and unique recruitment across any S are retained
in `recruitment` for inspection/export. The other metric cells still use all
four repetitions. Display axes are nominal; classifications use actual samples.

An additional Exp 7-style diagnostic shows the mean significant traces beside
a **binary neuron-by-phase matrix**: 1=active, 0=inactive (gray=missing). The
preconditioning column uses 12/15; each bout column uses 2/3. Mean trace pixels
remain fractions and are not thresholded into events. This display-only pooled
preconditioning criterion does not change the per-condition recruitment counts.


In [ ]:
# Cell 06b - Shared and newly recruited neurons, repetitions 2-4
RECRUITMENT_RASTER_FIGSIZE = (20, 9)
RECRUITMENT_COUNT_FIGSIZE = (13, 8)
RECRUITMENT_DECISION_FIGSIZE = (14, 8)
recruitment = diagnostics.build_phase_recruitment_analysis(
    cohort, analysis['trial_metrics'], RECRUITMENT_CONDITIONS,
    MEASUREMENT_SETTINGS['bout_windows'][RECRUITMENT_SETTINGS['bout_window']],
    **RECRUITMENT_SETTINGS)
for side in RECRUITMENT_CONDITIONS:
    FIGURES[f'recruitment_{side}_rasters'], _ = figures.plot_phase_recruitment_rasters(
        recruitment, side, figsize=RECRUITMENT_RASTER_FIGSIZE)
    plt.show()
    FIGURES[f'recruitment_{side}_decisions'], _ = figures.plot_phase_recruitment_decisions(
        recruitment, side, figsize=RECRUITMENT_DECISION_FIGSIZE)
    plt.show()
FIGURES['recruitment_counts'], _ = figures.plot_phase_recruitment_counts(
    recruitment, cohort['fish_ids'], RECRUITMENT_CONDITIONS,
    figsize=RECRUITMENT_COUNT_FIGSIZE)
plt.show()


## Cell 07 - Preconditioning, pre-bout activity and repetition consistency

Preconditioning measurements stop at its explicit offset; no response tail extends into the blank or bout. Early/late means use the first/second halves of that phase. Their difference describes temporal change, not automatically adaptation. The 2-second pre-bout level can include conditioning activity, especially at short delays; it is not a quiet baseline. Corrected bout AUC subtracts that descriptive ongoing level, without isolating a causal bout response. C1 preconditioning points are absent, not zero.


In [ ]:
# Cell 07 - Preconditioning, pre-bout activity and repetition consistency
summaries = [('preconditioning', 'auc_z_s', 'Preconditioning AUC (z s)'),
    ('preconditioning', 'peak_z', 'Preconditioning peak z-score'),
    ('preconditioning', 'active_fraction', 'Preconditioning active fraction'),
    ('preconditioning', 'early_late_change_z', 'Late minus early mean z-score'),
    ('pre_bout', 'mean_z', 'Ongoing pre-bout mean z-score')]
for window in MEASUREMENT_SETTINGS['bout_windows']:
    summaries += [(window, 'delta_auc_z_s', 'AUC above pre-bout level (z s)'),
                  (window, 'mean_active_trials', 'Mean passing trials per neuron (0-4)')]
for window, metric, units in summaries:
    name = f'{window}_{metric}'
    FIGURES[name], _ = figures.plot_phase_fish_summary(
        analysis['fish_summary'], metric, SIDE_CONDITIONS, cohort['fish_ids'],
        window, units, figsize=(14, 4.5))
    plt.show()

## Cell 08 - Paired reference comparisons and delay dependence

Paired differences compare each conditioned stimulus with same-side C1, plus C2 versus S1. Delay curves use the actual unequal blank delays on the x-axis. Both bout windows are shown separately; keep left/right responses distinct.


In [ ]:
# Cell 08 - Paired reference comparisons and delay dependence
comparison_figures = figures.plot_phase_population_comparisons(
    analysis, cohort['fish_ids'], SIDE_CONDITIONS, MEASUREMENT_SETTINGS['bout_windows'],
    COMPARISONS, DELAY_CONDITIONS, figsize=(14, 10))
FIGURES.update(comparison_figures)
plt.show()

## Cell 09 - Phase memberships and within-fish relationships

Fractions describe reliable neurons active in preconditioning only, bouts only, both or neither. C1 lacks the first phase and is omitted from these categories. Scatter points retain individual neuron/fish identity. Spearman correlations are descriptive within each fish, condition and bout window; insufficient or constant data produce missing correlations, not zero.


In [ ]:
# Cell 09 - Phase memberships and within-fish relationships
relationship_figures = figures.plot_phase_population_relationships(
    analysis, cohort['fish_ids'], SIDE_CONDITIONS, MEASUREMENT_SETTINGS['bout_windows'],
    membership_figsize=(14, 12), scatter_figsize=(20, 7))
FIGURES.update(relationship_figures)
plt.show()
display(analysis['correlations'])

## Cell 10 - Save this population run after review

Enable the final save only when satisfied. The existing report writer saves tables, configuration, readiness, original-neuron display mappings and figures under `<Results/Paper/2p_derived>/<experiment>/reports/<timestamp>_population_analysis/`. Individual-fish outputs are read-only. No files are written while saving is off.


In [ ]:
# Cell 10 - Save this population run after review
SAVE_OUTPUTS = False
TABLES = dict(analysis, readiness=cohort['readiness'],
    sequence_display_rows=displays['sequence']['displayed_row_metadata'],
    bout_display_rows=displays['bout']['displayed_row_metadata'])
TABLES.update({f'{anchor}_{scope}_display_rows': displays[anchor, scope]['displayed_row_metadata']
               for anchor in ['sequence', 'bout'] for scope in DISPLAY_SORT_SCOPES})
TABLES.update({f'recruitment_{name}': recruitment[name]
               for name in ['neurons', 'fish_summary', 'union_summary', 'preconditioning_pool']})
TABLES.update({f'recruitment_{side}_display_rows': bundle['displayed_row_metadata']
               for side, bundle in recruitment['displays'].items()})
run = reporting.save_analysis_report_run(
    settings=dict(COHORT_SETTINGS, measurement_settings=MEASUREMENT_SETTINGS,
                  recruitment_settings=RECRUITMENT_SETTINGS, recruitment_conditions=RECRUITMENT_CONDITIONS,
                  sort_mode=SORT_MODE, sort_stimulus_keys=SORT_STIMULUS_KEYS,
                  sort_alignment=SORT_ALIGNMENT, sort_window_s=SORT_WINDOW_S,
                  display_sort_scopes=DISPLAY_SORT_SCOPES,
                  display_repetitions=DISPLAY_REPETITIONS, raster_cmap=RASTER_CMAP,
                  zscore_cmap=ZSCORE_CMAP,
                  z_limits=Z_LIMITS, comparisons=COMPARISONS,
                  side_conditions=SIDE_CONDITIONS, delay_conditions=DELAY_CONDITIONS),
    report_settings=dict(save_report=SAVE_OUTPUTS, run_label='population_analysis', export_notebook=False),
    analysis_path=ANALYSIS_PATH, tables=TABLES, figures=FIGURES,
    extra_metadata=dict(included_fish=cohort['fish_ids'], single_fish_preview=len(cohort['fish_ids']) == 1,
        detector_settings={fid: fish['detector_settings'] for fid, fish in cohort['all_fish_data'].items()},
        recording_sessions={fid: fish['session_id'] for fid, fish in cohort['all_fish_data'].items()}))